# Day 19 SQL 业务分析预览

本 notebook 不连接 MySQL，只读取 `outputs/sql_exports/` 下的 CSV，模拟展示 Day19 SQL 业务分析要回答的问题。SQL 文件仍然是本轮主产物。

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from scania_aps.config import get_config

cfg = get_config(PROJECT_ROOT / 'config' / 'config.yaml')
sql_export_dir = cfg.project_root / 'outputs' / 'sql_exports'
sql_export_dir

WindowsPath('C:/Scania APS/outputs/sql_exports')

In [2]:
pred = pd.read_csv(sql_export_dir / 'model_prediction_results.csv')
policy = pd.read_csv(sql_export_dir / 'model_policy_comparison.csv')
thresholds = pd.read_csv(sql_export_dir / 'threshold_sensitivity_results.csv')

pred.shape, policy.shape, thresholds.shape

((16000, 17), (5, 17), (99, 15))

## Top-K 维修容量分析

观察如果只检查风险最高的 Top K 样本，可以覆盖多少真实 APS 故障。

In [3]:
total_pos = int(pred['y_true'].sum())
topk_rows = []
ranked = pred.sort_values('y_proba', ascending=False).reset_index(drop=True)
for k in [50, 100, 200, 500, 1000]:
    subset = ranked.head(k)
    actual_pos = int(subset['y_true'].sum())
    topk_rows.append({
        'top_k': k,
        'inspected_count': len(subset),
        'actual_pos_count': actual_pos,
        'precision_at_k': actual_pos / len(subset),
        'recall_at_k': actual_pos / total_pos,
        'missed_pos_count': total_pos - actual_pos,
    })
pd.DataFrame(topk_rows)

,top_k,inspected_count,actual_pos_count,precision_at_k,recall_at_k,missed_pos_count
0,50,50,50,1.000,0.133333,325
1,100,100,100,1.000,0.266667,275
2,200,200,194,0.970,0.517333,181
3,500,500,340,0.680,0.906667,35
4,1000,1000,369,0.369,0.984000,6


## 风险等级与维修工作量

In [4]:
risk_summary = (
    pred.groupby(['risk_level', 'suggested_action'], as_index=False)
    .agg(
        sample_count=('sample_id', 'count'),
        actual_pos_count=('y_true', 'sum'),
        predicted_pos_count=('y_pred', 'sum'),
        avg_predicted_probability=('y_proba', 'mean'),
        total_sample_cost=('sample_cost', 'sum'),
    )
)
risk_summary['actual_pos_rate'] = risk_summary['actual_pos_count'] / risk_summary['sample_count']
risk_summary

,risk_level,suggested_action,sample_count,actual_pos_count,predicted_pos_count,avg_predicted_probability,total_sample_cost,actual_pos_rate
0,Critical,immediate_inspection,404,316,404,0.962373,880,0.782178
1,High,priority_inspection,321,46,321,0.454767,2750,0.143302
2,Low,no_action_now,14861,5,0,0.003503,2500,0.000336
3,Medium,monitor_and_recheck,414,8,36,0.105079,3850,0.019324


## 错误分析

In [5]:
error_summary = (
    pred.groupby('confusion_type', as_index=False)
    .agg(sample_count=('sample_id', 'count'), avg_probability=('y_proba', 'mean'), total_cost=('sample_cost', 'sum'))
)
error_summary

,confusion_type,sample_count,avg_probability,total_cost
0,FN,12,0.067298,6000
1,FP,398,0.522033,3980
2,TN,15227,0.005773,0
3,TP,363,0.919758,0


## 成本策略对比

In [6]:
policy.sort_values(['dataset', 'total_cost'])

,policy_name,dataset,threshold,fp,fn,tp,tn,fp_cost,fn_cost,fp_cost_total,fn_cost_total,total_cost,baseline_total_cost,cost_reduction,cost_reduction_rate,source_file,note
2,day14_structural_all_final_candidate,official_test,0.18,398,12,363,15227,10,500,3980,6000,9980,187500,177520,0.946773,outputs/metrics/day14_structural_feature_test_...,当前最终推荐候选，使用 Day13 valid 选择的 threshold=0.18。
1,day14_baseline_median_all,official_test,0.16,432,13,362,15193,10,500,4320,6500,10820,187500,176680,0.942293,outputs/metrics/day14_structural_feature_test_...,Day14 未调参 baseline_median_all official test 结果。
3,day16_tuned_best,official_test,0.19,376,15,360,15249,10,500,3760,7500,11260,187500,176240,0.939947,outputs/metrics/day16_xgb_tuning_test_results.csv,Day15 tuned 参数在 Day16 official test 上的最低成本方案，用...
0,naive_all_negative,official_test,NaN,0,375,0,15625,10,500,0,187500,187500,187500,0,0.000000,outputs/predictions/day14_structural_feature_t...,全预测为 neg 的 naive baseline，作为 official test 成本下...
4,day18_oof_ensemble_best,oof_train,0.14,1841,41,959,57159,10,500,18410,20500,38910,500000,461090,0.922180,outputs/metrics/day18_oof_ensemble_best_summar...,OOF train 内部口径结果，不可与 official test 成本直接横向比较。


## Decile / Lift / Gain

In [7]:
overall_pos_rate = pred['y_true'].mean()
decile = pred.groupby('decile', as_index=False).agg(
    sample_count=('sample_id', 'count'),
    actual_pos_count=('y_true', 'sum'),
)
decile['pos_rate'] = decile['actual_pos_count'] / decile['sample_count']
decile['cumulative_pos_count'] = decile['actual_pos_count'].cumsum()
decile['cumulative_recall'] = decile['cumulative_pos_count'] / total_pos
decile['lift'] = decile['pos_rate'] / overall_pos_rate
decile

,decile,sample_count,actual_pos_count,pos_rate,cumulative_pos_count,cumulative_recall,lift
0,1,1600,373,0.233125,373,0.994667,9.946667
1,2,1600,0,0.000000,373,0.994667,0.000000
2,3,1600,1,0.000625,374,0.997333,0.026667
3,4,1600,0,0.000000,374,0.997333,0.000000
4,5,1600,0,0.000000,374,0.997333,0.000000
5,6,1600,1,0.000625,375,1.000000,0.026667
6,7,1600,0,0.000000,375,1.000000,0.000000
7,8,1600,0,0.000000,375,1.000000,0.000000
8,9,1600,0,0.000000,375,1.000000,0.000000
9,10,1600,0,0.000000,375,1.000000,0.000000


## 阈值敏感性

该分析只用于业务策略敏感性展示，不用于修改最终方案 threshold=0.18。

In [8]:
thresholds.sort_values(['total_cost', 'fn', 'recall_score'], ascending=[True, True, False]).head(10)

,threshold,predicted_positive_count,predicted_negative_count,tp,fp,tn,fn,precision_score,recall_score,f1_score,f2_score,fp_cost,fn_cost,total_cost,workload_rate
6,0.07,1016,14984,369,647,14978,6,0.363189,0.984000,0.530554,0.733307,10,500,9470,0.063500
7,0.08,981,15019,368,613,15012,7,0.375127,0.981333,0.542773,0.741636,10,500,9630,0.061312
8,0.09,946,15054,367,579,15046,8,0.387949,0.978667,0.555640,0.750204,10,500,9790,0.059125
14,0.15,802,15198,364,438,15187,11,0.453865,0.970667,0.618522,0.790617,10,500,9880,0.050125
21,0.22,703,15297,362,341,15284,13,0.514936,0.965333,0.671614,0.821607,10,500,9910,0.043937
17,0.18,761,15239,363,398,15227,12,0.477004,0.968000,0.639085,0.802742,10,500,9980,0.047563
5,0.06,1068,14932,369,699,14926,6,0.345506,0.984000,0.511435,0.718458,10,500,9990,0.066750
20,0.21,711,15289,362,349,15276,13,0.509142,0.965333,0.666667,0.818634,10,500,9990,0.044437
13,0.14,820,15180,364,456,15169,11,0.443902,0.970667,0.609205,0.784483,10,500,10060,0.051250
3,0.04,1231,14769,372,859,14766,3,0.302193,0.992000,0.463263,0.681069,10,500,10090,0.076938


## Day 19 小结

- 本轮主产物是 SQL 和 MySQL 导入 CSV，不是新的模型结果。
- `sample_id` 是匿名样本编号，不是真实车辆 ID。
- OOF 口径和 official test 口径不能直接横向比较。
- 阈值敏感性用于解释业务权衡，不用于回头修改最终阈值。